# The Friendly Fraudsters | [Company]: Card Dispute Agent

FSI Hackathon London, 25 November 2026

## 1. The problem

UK banks handle thousands of card disputes: *"I didn't make this payment"*, *"I cancelled and they still charged me"*, *"my sofa never arrived"*. For each one, a disputes analyst works out what kind of problem it is, looks up the payment and the customer's history, decides who is right, writes an evidence pack for the chargeback and writes to the customer. It is slow, repetitive work, and vulnerable customers need extra care under the FCA's Consumer Duty.

**Who it helps:** the bank's disputes operations team, and the customers waiting for an answer.

**What we built:** a LangGraph team of agents that checks vulnerability, classifies the dispute, routes it to a fraud, merchant or friendly-fraud specialist, drafts the evidence pack, has a quality reviewer check it, pauses for a human to approve, writes the customer letter and remembers the outcome.

## 2. Setup

Run this notebook from the `the_friendly_fraudsters/` folder, with the repo's `.env` filled in (see the README). All data is synthetic.

In [1]:
import json
from collections import Counter
from IPython.display import Markdown, display

from utils import db

complaints = db.table("complaints")
print(f"{len(complaints)} complaints, {len(db.table('customers'))} customers, "
      f"{len(db.table('transactions'))} transactions, {len(db.table('dispute_history'))} past disputes")

12 complaints, 8 customers, 19 transactions, 3 past disputes


## 3. The dataset

Every complaint is labelled with the correct reason code, route, recommendation and vulnerability level, plus a one-line note on why. The cases cover every path through the agent, including traps: a vague "I don't recognise this", an angry customer who is *not* vulnerable, and a crisis case that must go straight to a human.

In [2]:
rows = ["| Case | Customer says | Reason code | Route | Vulnerability |", "| --- | --- | --- | --- | --- |"]
for c in complaints:
    text = c["text"] if len(c["text"]) < 90 else c["text"][:87] + "..."
    rows.append(f"| {c['complaint_id']} | {text} | {c['expected_reason_code']} | {c['expected_route']} | {c['expected_vulnerability']['level']} |")
display(Markdown("\n".join(rows)))

| Case | Customer says | Reason code | Route | Vulnerability |
| --- | --- | --- | --- | --- |
| CMP-001 | I didn't make this payment to GadgetHub. I was asleep at 3am and I've never shopped the... | fraud_card_not_present | fraud | none |
| CMP-002 | I cancelled StreamMax last month but they charged me again. | cancelled_recurring | merchant | none |
| CMP-003 | I paid for a sofa three weeks ago and it never arrived. The shop won't answer my calls ... | goods_not_received | merchant | medium |
| CMP-004 | This PixelGames charge isn't me, I never bought anything. | fraud_card_not_present | friendly_fraud | none |
| CMP-005 | My wallet was stolen on Saturday afternoon in central London and someone used my card a... | fraud_card_present | fraud | none |
| CMP-006 | I don't recognise this payment to PAYPAL *DIGIMARKT. What is it? | fraud_card_not_present | fraud | none |
| CMP-007 | RailGo took the money for my train ticket twice on the same morning. | duplicate_charge | merchant | none |
| CMP-008 | Bella Cucina charged me £148 but our bill was £48. I think they added an extra 1 by mis... | incorrect_amount | merchant | none |
| CMP-009 | I never ordered from FoodDash on the 20th, this isn't mine. | fraud_card_not_present | friendly_fraud | none |
| CMP-010 | I'm 82 and I don't really understand online payments, my grandson set it up for me. A m... | fraud_card_not_present | fraud | medium |
| CMP-011 | Someone took £600 from my account last night and now I can't pay my rent. I can't cope ... | fraud_card_not_present | fast_lane | high |
| CMP-012 | I'm absolutely furious! I cancelled StreamMax in October and they've charged me AGAIN. ... | cancelled_recurring | merchant | none |

In [3]:
print("Routes:", dict(Counter(c["expected_route"] for c in complaints)))
print("Vulnerability:", dict(Counter(c["expected_vulnerability"]["level"] for c in complaints)))
print("Reason codes:", dict(Counter(c["expected_reason_code"] for c in complaints)))

Routes: {'fraud': 4, 'merchant': 5, 'friendly_fraud': 2, 'fast_lane': 1}
Vulnerability: {'none': 9, 'medium': 2, 'high': 1}
Reason codes: {'fraud_card_not_present': 6, 'cancelled_recurring': 2, 'goods_not_received': 1, 'fraud_card_present': 1, 'duplicate_charge': 1, 'incorrect_amount': 1}


### Data checks

Every complaint must point to a real customer and payment, and the triage rules must agree with every route label.

In [4]:
from evals.check_data import check
problems = check()
print("\n".join(problems) if problems else "All good: every case fits together.")

All good: every case fits together.


### Test sets for the quality reviewer and the letters

- **Quality reviewer:** 8 evidence packs, 2 good and 6 with one planted mistake (a made-up fact, a recommendation that contradicts the evidence, a missing section, ignored vulnerability, an accusing tone, a wrong reason code).
- **Customer letters:** 8 scenarios with rules for what each letter must and must not say, for example a decline must mention the Financial Ombudsman Service, and a friendly-fraud letter must never accuse.

In [5]:
from evals.test_sets import qa_packs, letter_cases
for p in qa_packs():
    print(f"{p['pack_id']}  {p['complaint_id']}  expected: {p['expected_verdict']:<4}  planted issue: {p['expected_issue']}")
print()
for l in letter_cases():
    print(f"{l['letter_id']}  {l['complaint_id']}  {l['outcome']:<26} vulnerable={l['vulnerable']}")

QA-01  CMP-003  expected: pass  planted issue: none
QA-02  CMP-001  expected: pass  planted issue: none
QA-03  CMP-001  expected: fail  planted issue: made_up_fact
QA-04  CMP-004  expected: fail  planted issue: recommendation_mismatch
QA-05  CMP-007  expected: fail  planted issue: missing_section
QA-06  CMP-010  expected: fail  planted issue: vulnerability_ignored
QA-07  CMP-009  expected: fail  planted issue: accusing_tone
QA-08  CMP-002  expected: fail  planted issue: wrong_reason_code

LTR-01  CMP-001  file_chargeback            vulnerable=False
LTR-02  CMP-003  file_chargeback            vulnerable=True
LTR-03  CMP-004  decline_claim              vulnerable=False
LTR-04  CMP-009  decline_claim              vulnerable=False
LTR-05  CMP-010  file_chargeback            vulnerable=True
LTR-06  CMP-002  needs_more_info            vulnerable=False
LTR-07  CMP-012  file_chargeback            vulnerable=False
LTR-08  CMP-011  fast_lane_holding_message  vulnerable=True


## 4. One case end to end

*To add after the quality reviewer and customer letter agents are merged.*

## 5. Evals

*Baseline (before the new agents), 12 cases: reason code 12/12, route 11/12, vulnerability flag 12/12, high-risk cases caught 0/1. The one miss is CMP-011: the agent spotted the vulnerability but had no fast lane yet, so the crisis case went through normal fraud handling.*

## 6. Results table

*To add.*

## 7. KPIs and ROI

*To add.*

## 8. Next steps

*To add.*